<a href="https://colab.research.google.com/github/MelB18/EasyQuanten/blob/main/myMolM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
%pip install --upgrade 'qiskit[visualization]>=2.5.0' qiskit-aer qiskit_addon_sqd

In [25]:
%pip install rdkit py3Dmol pyscf ffsim

In [31]:
# --- IMPORTE: laden aller benötigten Werkzeuge ---
import numpy as np # Bibliothek für mathematische Arrays und Zahlen
import pyscf # Klassisches Chemie-Programm zur Berechnung der Molekül-Basisdaten
import pyscf.cc # Modul für die klassische CCSD-Schätzung (Startwerte für den Schaltkreis)
import pyscf.fci # Modul für die exakte Vergleichslösung (zur Kontrolle am Ende)
import ffsim # Spezialbibliothek zum Bauen von Quantenchemie-Schaltkreisen
from qiskit import QuantumCircuit, QuantumRegister # Grundbausteine für Qiskit-Schaltkreise
from qiskit.primitives import StatevectorSampler # Lokaler Simulator, der einen Quantencomputer nachahmt



In [32]:
from qiskit_addon_sqd.fermion import ( # IBMs neues SQD-Addon für die Auswertung
  SCIResult,
  diagonalize_fermionic_hamiltonian,
  solve_sci_batch,
)

# Moleküle definieren


PySCF (Python-based Simulations of Chemistry Framework):

.build() ist eine Methode des gto.Mole Objekts, um die Molekülstruktur zu initialisieren.

Eingabeparameter:

* Atome im 3D-Raum "atom": aus welchen Elementen besteht das Atom und wo im Raum sitzen die Atomkerne.

* Elektronen-Wolken "basis":
Mit dem Basissatz legt man fest wie feinmaschig die Orbitale der Elektronen beschrieben werden.


größerer Basissatz => mehr Qubits => genauere Ergebnisse






# Wasserstoffmolekül

Ziel: Berechnung der Form und Energie der Elektronenwolken (Orbitale) eines Moleküls berechnen.

Da sich die exakte Schrödinger-Gleichung für Moleküle nicht einfach lösen lässt, baut man die komplizierten Elektronenwolken aus vielen einfachen, glockenförmigen Bausteinen zusammen.

Bauklotz-Analogie:

* komplizierte Skulptur (das echte Molekülorbital)

* nur runde Bauklötze (mathematische Funktionen)


Wenig Bauklötze (z. B. STO-3G): Die Skulptur wird sehr grob und eckig. Die Rechnung geht extrem schnell, aber das Ergebnis (z. B. die Bindungslänge oder Energie) ist oft ungenau.

Viele Bauklötze (z. B. 6-311++G(3df,3pd)): Die Skulptur wird extrem detailliert und realistisch. Allerdings braucht der Computer dafür sehr viel Rechenzeit und Arbeitsspeicher.

Der goldene Mittelweg (6-31G): Dieser Basissatz bietet genau die richtige Menge an Bausteinen, um verlässliche chemische Ergebnisse zu bekommen, ohne den Computer zu überlasten.

In [33]:

# Wir definieren das Wasserstoffmolekül (H2) mit zwei Atomen im Abstand von 0.74 Angström
mol = pyscf.gto.Mole()
mol.build(
  atom=[["H", (0.0, 0.0, 0.0)], ["H", (0.0, 0.0, 0.74)]], # Positionen der zwei H-Atome im Raum
  basis="6-31g" # Genauigkeitsstufe der Elektronen-Orbitale (ergibt 4 Orbitale = 8 Qubits)
)

In [35]:
mol = pyscf.gto.Mole()
mol.build(
    atom=[["H", (0.0, 0.0, 0.0)], ["H", (0.0, 0.0, 0.74)]], # Positionen der zwei H-Atome im Raum
    basis="6-31G"
)

Ein Wasserstoff-Atom (H) ist das einfachste Beispiel, um 6-31G zu verstehen, weil es nur 1 Elektron im 1s-Orbital besitzt. Da Wasserstoff gar keine inneren Rumpfelektronen hat, fällt der vordere Teil („6“) der Bezeichnung hier komplett weg.

Wie beschreibt 6-31G ein Wasserstoff-Atom?

Das einzige 1s-Orbital des Wasserstoffs ist gleichzeitig sein Valenzorbital (es geht die chemische Bindung ein). Deshalb wendet 6-31G sofort die 3-1-Aufteilung (Split-Valence) an:

* Keine Rumpfelektronen („6“ entfällt): Weil es kein tiefer liegendes Orbital unter dem 1s-Orbital gibt, wird die „6“ für Wasserstoff nicht benötigt.
* Erste Basisfunktion („3“): Das 1s-Orbital bekommt einen kompakten, kernnahen Baustein. Dieser besteht aus 3 fest zusammengeklebten Gauß-Funktionen (GTOs), die das Elektron dicht am Proton beschreiben.
* Zweite Basisfunktion („1“): Zusätzlich bekommt das 1s-Orbital einen zweiten, weitläufigeren Baustein. Dieser besteht aus 1 einzelnen Gauß-Funktion, die den äußeren Rand der Elektronenwolke abdeckt.

Flexibilität bei Bindungen: Der Computer kann beide Bausteine nun beliebig mischen. Geht der Wasserstoff eine Bindung mit einem stark ziehenden Atom (wie Sauerstoff in H₂O) ein, zieht sich die Wolke durch einen höheren Anteil der „3er-Funktion“ etwas zusammen.

# Lithiumhydrid

In [ ]:
# LiH mit sto-3g Basis benötigt 12 Qubits (6 Orbitale, 4 Elektronen)
# experimenteller Bindungsabstand von 1.595 Angström
mol = pyscf.gto.Mole()
mol.build(
  atom=[
      ["Li", (0.0, 0.0, 0.0)],
      ["H", (0.0, 0.0, 1.595)]
  ], # Positionen der Atome im Raum
  basis="sto-3g" # Genauigkeitsstufe der Elektronen-Orbitale (ergibt 6 Orbitale = 12 Qubits)
)

NameError: name 'pyscf' is not defined

 # Wasser

In [ ]:
# Wassermolekül (H2O) in seiner gewinkelten Gleichgewichtsgeometrie

mol = pyscf.gto.Mole()
mol.build(
    atom=[
      ["O", (0.000, 0.000, 0.117)],
      ["H", (0.000, 0.757, -0.469)],
      ["H", (0.000, -0.757, -0.469)],
    ], # Positionen der Atome im Raum
    basis="sto-3g" # Genauigkeitsstufe der Elektronen-Orbitale (ergibt 7 Orbitale = 14 Qubits)
)

Positionsdaten mit Hilfe von rdkit ermitteln und Moleküle visualisieren.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
import py3Dmol


# Molekül aus SMILES-Code erstellen (O steht für Wasser)
rkmol = Chem.MolFromSmiles("O")

# Wasserstoffatome explizit hinzufügen (für H2O statt nur O)
mol_with_h = Chem.AddHs(rkmol)

# Grundlegende Eigenschaften berechnen
mol_weight = Descriptors.MolWt(rkmol)
# durchschnittliche Molekülmasse
# inklusive aller impliziten Wasserstoffatome
num_atoms_implizit = rkmol.GetNumAtoms()
num_atoms_explizit = mol_with_h.GetNumAtoms()

print(f"Molekülformel: {Chem.rdMolDescriptors.CalcMolFormula(rkmol)}")
print(f"Molekulargewicht: {mol_weight:.3f} g/mol")
print(f"Atome (ohne explizites H): {num_atoms_implizit}")
print(f"Atome (mit explizitem H): {num_atoms_explizit}")

# 3D-Koordinaten generieren
AllChem.EmbedMolecule(mol_with_h, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol_with_h) # Kraftfeld-Optimierung

# Koordinaten für PySCF berechnen
conf = mol_with_h.GetConformer()
atom_liste = []
for atom in mol_with_h.GetAtoms():
    pos = conf.GetAtomPosition(atom.GetIdx())
    atom_liste.append([atom.GetSymbol(), (pos.x, pos.y, pos.z)])
print(atom_liste)

# Daten in PySCF laden
mol = pyscf.gto.Mole()
mol.build(atom=atom_liste, basis="sto-3g")

# In MolBlock-Text umwandeln und im 3D-Viewer laden
mol_block = Chem.MolToMolBlock(mol_with_h)
viewer = py3Dmol.view(width=400, height=300)
viewer.addModel(mol_block, "mol")

# Darstellungsstil wählen (Kugel-Stab-Modell)
viewer.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
viewer.zoomTo()
viewer.show()


Molekülformel: H2O
Molekulargewicht: 18.015 g/mol
Atome (ohne explizites H): 1
Atome (mit explizitem H): 3
[['O', (0.007827036328440498, 0.39773799122489173, 0.0)], ['H', (-0.7672341344362831, -0.18384734761231294, 0.0)], ['H', (0.7594070981078427, -0.21389064361257837, 0.0)]]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Berechnen

In [ ]:
# Wir führen eine erste grobe klassische Berechnung durch (Hartree-Fock-Methode)
scf = pyscf.scf.RHF(mol).run()

# Wir lesen die wichtigsten physikalischen Werte aus dem Molekül aus:
num_orbitals = scf.mo_coeff.shape[1] # Anzahl der räumlichen Orbitale (hier: 4)
print(f"Anzahl der Orbitale: {num_orbitals}")

nelec = mol.nelec # Anzahl der Elektronen als Paar: (1 Alpha-Elektron, 1 Beta-Elektron)
print(f"Anzahl der Elektronen: {nelec}")

nuclear_repulsion_energy = mol.energy_nuc() # Abstoßungsenergie der beiden Atomkerne
print(f"Abstoßungsenergie: {nuclear_repulsion_energy}")


# Wir berechnen die Wechselwirkungs-Integrale der Elektronen (das "Spielfeld" der Kräfte)
hcore = scf.mo_coeff.T @ scf.get_hcore() @ scf.mo_coeff # Ein-Elektronen-Integrale (Bewegung + Kernanziehung)
eri = pyscf.ao2mo.kernel(mol, scf.mo_coeff) # Zwei-Elektronen-Integrale (gegenseitige Abstoßung)
eri = pyscf.ao2mo.restore(1, eri, num_orbitals) # Bringt die Abstoßungsdaten in ein 4D-Tabellenformat

print(f"Shape der Ein-Elektronen-Integrale: {hcore.shape}")
print(f"Shape der Zwei-Elektronen-Integrale: {eri.shape}")

# Wir machen eine klassische CCSD-Schätzung, um gute Start-Parameter für den Schaltkreis zu erhalten
ccsd = pyscf.cc.CCSD(scf).run()

converged SCF energy = -74.9644491251886
Anzahl der Orbitale: 7
Anzahl der Elektronen: (5, 5)
Abstoßungsenergie: 9.084262014125592
Shape der Ein-Elektronen-Integrale: (7, 7)
Shape der Zwei-Elektronen-Integrale: (7, 7, 7, 7)
E(CCSD) = -75.01541234323719  E_corr = -0.05096321804856783
